# Doctor Recommendation: Segmented Classification Approach

This notebook implements a segmented machine learning system using **classification** to predict the probability of a 'Good Match' between a patient and a doctor. It trains separate expert models (e.g., LightGBM, XGBoost) for different patient preference groups and ranks recommendations based on predicted probability.

### Step 1: Setup and Data Loading

In [11]:
import pandas as pd
import numpy as np
import pgeocode
import joblib
import json
from datetime import datetime
import os

# ML Imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import MinMaxScaler

# Import Classifiers
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

# Optional: Install libraries if needed
# !pip install pgeocode lightgbm xgboost openpyxl

try:
    import pgeocode
except ImportError:
    print("Installing pgeocode...")
    subprocess.run([sys.executable, "-m", "pip", "install", "pgeocode"], check=True)
    import pgeocode

try:
    from ethnicolr import census_ln
except ImportError:
    print("Installing ethnicolr...")
    # Note: ethnicolr requires TensorFlow. This installation might take a moment.
    subprocess.run([sys.executable, "-m", "pip", "install", "ethnicolr"], check=True)
    from ethnicolr import census_ln

import warnings
#warnings.filterwarnings("ignore", category=FutureWarning)

import os
from sklearn.model_selection import ParameterGrid, cross_validate
from sklearn.model_selection import ParameterSampler
import joblib

try:
    import lightgbm as lgb
except ImportError:
    print("Installing lightgbm...")
    subprocess.run([sys.executable, "-m", "pip", "install", "lightgbm"], check=True)
    import lightgbm as lgb

try:
    import xgboost as xgb
except ImportError:
    print("Installing xgboost...")
    subprocess.run([sys.executable, "-m", "pip", "install", "xgboost"], check=True)
    import xgboost as xgb

try:
    import catboost as cb
except ImportError:
    print("Installing catboost...")
    subprocess.run([sys.executable, "-m", "pip", "install", "catboost"], check=True)
    import catboost as cb

try:
    import gender_guesser.detector as gender
except ImportError:
    print("Installing gender-guesser...")
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "gender-guesser"], check=True)
    import gender_guesser.detector as gender



In [12]:
# --- Data Loading ---
#file locations
folder_path = r"Client_Data_files\Parquets"
parquet_file_paths={
    "patient": os.path.join(folder_path, "synthetic_patients.parquet"),
    "encounter": os.path.join(folder_path,"synthetic_encounters.parquet"),
    "hospitals": os.path.join(folder_path,"synthetic_hospitals.parquet"),
    "provider": os.path.join(folder_path,"synthetic_providers.parquet"),    
}

# Reading the parquet files
patient_df = pd.read_parquet(parquet_file_paths['patient'])
encounter_df = pd.read_parquet(parquet_file_paths['encounter'])
hospital_df = pd.read_parquet(parquet_file_paths['hospitals'])
provider_df = pd.read_parquet(parquet_file_paths['provider'])

print("Dataframes loaded successfully.")
print(f"Patient DF shape: {patient_df.shape}")
print(f"Encounter DF shape: {encounter_df.shape}")
print(f"Provider DF shape: {provider_df.shape}")
print(f"Hospital DF shape: {hospital_df.shape}")
print("Dataframes loaded successfully.")

Dataframes loaded successfully.
Patient DF shape: (100000, 16)
Encounter DF shape: (200000, 17)
Provider DF shape: (5000, 19)
Hospital DF shape: (200, 14)
Dataframes loaded successfully.


### Step 2: Initial Data Merging and Stateless Feature Engineering

In [13]:
# Creating a data map for easy access
data_map={
    "patient": patient_df,
    "encounter": encounter_df,
    "hospitals": hospital_df,
    "provider": provider_df
}

def describe_Cols(data_map):
    for key, df in data_map.items():
        print(f"Dataframe: {key}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")    
        for col in df.columns:
            if df[col].isna().sum() > 0:
                print(f"Column '{col}' has {df[col].isna().sum()} missing values.")
        print("")

In [14]:
describe_Cols(data_map)

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

In [15]:
race_mapping={
    'white': 'White',
    'black': 'Black or African American',
    'api': 'Asian',    
    'aian': 'Native American',
    '2prace': 'Other'
}

In [16]:
patient_df_temp=pd.DataFrame()
patient_df_temp=patient_df[patient_df['race']=='Hispanic or Latino'][['patient_id','first_name','last_name','race','ethnicity']].copy()
patient_race_pred=census_ln(patient_df_temp, 'last_name')

race_cols=['pctwhite','pctblack','pctapi','pctaian','pct2prace']
patient_race_pred['derived_race'] = patient_race_pred[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)

# Create a mapping from patient_id to derived_race
id_to_derived_race = dict(zip(patient_race_pred['patient_id'], patient_race_pred['derived_race']))

# Update the race column only for Hispanic or Latino patients
patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'race'] = \
    patient_df.loc[patient_df['race'] == 'Hispanic or Latino', 'patient_id'].map(id_to_derived_race)

2025-10-19 13:13:06,669 - INFO - Preserving 12965 duplicate rows based on column 'last_name'
2025-10-19 13:13:06,670 - INFO - Data filtering summary: 12997 → 12997 rows (kept 100.0%)
2025-10-19 13:13:06,671 - INFO - Loading Census 2000 data from c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\ethnicolr\data\census\census_2000.csv...
2025-10-19 13:13:06,825 - INFO - Loaded 151670 last names from Census 2000
2025-10-19 13:13:06,825 - INFO - Merging demographic data for 12997 records...
2025-10-19 13:13:06,863 - INFO - Matched 12997 of 12997 rows (100.0%)
2025-10-19 13:13:06,863 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


In [17]:
provider_race_predictions = census_ln(provider_df, 'last_name')

# Derive race for the provider_df as it is missing from the source data
print("Deriving race for providers from last names...")
race_cols = ['pctwhite','pctblack','pctapi','pctaian','pct2prace']
provider_df['provider_race'] = provider_race_predictions[race_cols].idxmax(axis=1).str.replace('pct', '').map(race_mapping)
print("Provider race derivation complete.")

# Deriving provider ethnicity from the race_predictions
print("Deriving ethnicity for providers from race predictions...")
provider_df['provider_ethnicity'] = provider_race_predictions['pcthispanic'].apply(lambda x: 'Hispanic or Latino' if float(x) >= 50 else 'Not Hispanic or Latino')
print("Provider ethnicity derivation complete.")


# Deriving provider gender from first names

# Initialize the gender detector 
gender_detector = gender.Detector()

def guess_gender(name):
    if pd.isna(name):
        return 'unknown'
    # Use 'unknown' as default, specify country if possible (e.g., 'usa')
    guessed = gender_detector.get_gender(str(name).split()[0]) # Use only the first word
    if guessed in ['male', 'mostly_male']:
        return 'male'
    elif guessed in ['female', 'mostly_female']:
        return 'female'
    else: # andy, unknown
        return 'unknown'
    
# Apply the function to the 'first_name' column of your provider_df

provider_df['derived_gender'] = provider_df['first_name'].apply(guess_gender)
# print(provider_df[['provider_id', 'first_name', 'derived_gender']].head())

2025-10-19 13:13:06,907 - INFO - Preserving 4968 duplicate rows based on column 'last_name'
2025-10-19 13:13:06,907 - INFO - Data filtering summary: 5000 → 5000 rows (kept 100.0%)
2025-10-19 13:13:06,916 - INFO - Merging demographic data for 5000 records...
2025-10-19 13:13:06,976 - INFO - Matched 5000 of 5000 rows (100.0%)
2025-10-19 13:13:06,976 - INFO - Added columns: pct2prace, pctaian, pctapi, pctblack, pcthispanic, pctwhite


Deriving race for providers from last names...
Provider race derivation complete.
Deriving ethnicity for providers from race predictions...
Provider ethnicity derivation complete.


In [18]:
describe_Cols(data_map)

Dataframe: patient
Shape: (100000, 16)
Columns: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background', 'preferred_provider_language', 'cultural_preferences']

Dataframe: encounter
Shape: (200000, 17)
Columns: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

Dataframe: hospitals
Shape: (200, 14)
Columns: ['hospital_id', 'hospital_name', 'hospital_type', 'zip_code', 'bed_count', 'teaching_hospital', 'trauma_center', 'language_services_available', 'cultural_competency_program', 'interpreter_services_24_7', 'community_health_programs', 'overall_rating

### Step 2: Feature Engineering

Here, we merge the datasets and create the features our model will learn from. This includes cultural matches, language matches, and geographic distance.

In [19]:
# Merge all data into a single master DataFrame for training
master_df = pd.merge(encounter_df, patient_df, on='patient_id',suffixes=('', '_pat'))
master_df = pd.merge(master_df, provider_df, on='provider_id',suffixes=('', '_prov'))
master_df = pd.merge(master_df, hospital_df, left_on='hospital_affiliation', right_on='hospital_id',suffixes=('', '_hosp'))

print("Master DataFrame created with shape:", master_df.shape)

Master DataFrame created with shape: (200000, 67)


In [20]:
df_order_list=['encounter', 'patient', 'provider', 'hospitals',]
prev=0
curr=0
for  i, df_name in enumerate(df_order_list, start=0):
    curr=curr+data_map[df_name].shape[1]-([0,1,1,4][i] if i < len([0,1,3,4]) else 0)
    print(f"{df_name}: {master_df.columns[prev:curr].tolist()}")
    print()
    prev=curr

encounter: ['encounter_id', 'patient_id', 'provider_id', 'encounter_date', 'encounter_type', 'primary_diagnosis', 'length_of_stay', 'total_cost', 'cultural_background', 'primary_language', 'languages_spoken', 'cultural_competency_rating', 'cultural_match_score', 'language_match', 'patient_satisfaction', 'treatment_adherence', 'return_visit_30_days']

patient: ['first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'primary_language_pat', 'zip_code', 'insurance_type', 'household_income', 'education_level', 'age', 'cultural_background_pat', 'preferred_provider_language', 'cultural_preferences']

provider: ['npi_number', 'first_name_prov', 'last_name_prov', 'specialty', 'practice_zip_code', 'years_experience', 'medical_school_country', 'board_certified', 'languages_spoken_prov', 'interpreter_services', 'cultural_certifications', 'minority_health_experience', 'community_involvement', 'patient_satisfaction_score', 'communication_rating', 'cultural_competency_rating_prov'

In [21]:
master_df['race_comb'] = (
    master_df['race'].fillna('Unknown') + 
    '-' + 
    master_df['provider_race'].fillna('Unknown')
)

print("Created race combination column:")
print(master_df['race_comb'].value_counts())

Created race combination column:
race_comb
White-White                                            107602
White-Asian                                             31983
Black or African American-White                         18589
Asian-White                                             12625
White-Black or African American                          9449
Black or African American-Asian                          5592
Asian-Asian                                              3621
Other-White                                              2857
Native American-White                                    2682
Black or African American-Black or African American      1694
Asian-Black or African American                          1119
Native American-Asian                                     853
Other-Asian                                               815
Other-Black or African American                           267
Native American-Black or African American                 252
Name: count, dtype: int64


In [22]:

# --- Engineer the Stateless Features ---

master_df['encounter_date'] = pd.to_datetime(master_df['encounter_date'])

master_df['board_certified'] = master_df['board_certified'].astype(int)
master_df['minority_health_experience'] = master_df['minority_health_experience'].astype(int)

# Stateless Cultural Features
master_df['race_match'] = (master_df['race'] == master_df['provider_race']).astype(int)
master_df['ethnicity_match'] = (master_df['ethnicity'] == master_df['provider_ethnicity']).astype(int)
master_df['language_match'] = (master_df['language_match'] == True).astype(int)
master_df['gender_match'] = (master_df['gender'] == master_df['derived_gender']).astype(int)





# Geographic Feature
dist = pgeocode.GeoDistance('US') # Assuming US zip codes
# Calculate distance between patient and provider zip codes
master_df['distance_km'] = dist.query_postal_code(
    master_df['zip_code'].astype(str).tolist(), 
    master_df['zip_code_hosp'].astype(str).tolist()
)
# Calculate the mean distance for each provider specialty
# The .transform('mean') creates a Series with the same index as master_df,
mean_dist_by_specialty = master_df.groupby('specialty')['distance_km'].transform('mean')

# Now, fill the missing distances using these specialty-specific averages
master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)

# If any specialties had NO valid distances, there might still be NaNs.
# Fill any remaining with the overall mean as a final fallback.
master_df['distance_km'].fillna(master_df['distance_km'].mean(), inplace=True)


C:\Users\jerry\AppData\Local\Temp\ipykernel_29632\2391335959.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  master_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)
C:\Users\jerry\AppData\Local\Temp\ipykernel_29632\2391335959.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

### Step 3: Configuration Block
Define models, parameters, features, and the threshold for a 'Good Match'.

In [23]:
# --- Target Definition --- 
# Define the threshold for a 'Good Match' based on composite score
scaler = MinMaxScaler()
master_df[['satisfaction_norm', 'adherence_norm']] = scaler.fit_transform(master_df[['patient_satisfaction', 'treatment_adherence']])
master_df['success_score'] = (master_df['adherence_norm'] * 0.5 + master_df['satisfaction_norm'] * 0.5)
GOOD_MATCH_THRESHOLD = master_df['success_score'].quantile(0.60) # e.g., Top 40% are 'Good'
target = 'is_good_match' # Our binary target column name

# --- Feature Definition --- 
features = [
    'years_experience', 'cultural_competency_rating_prov', 'communication_rating',
    'race_match', 'ethnicity_match', 'language_match', 'proximity_score', 
    'interpreter_services_24_7', 'historical_avg_adherence'
]

# --- Model Configuration --- 
model_configs = {
    # 'RandomForest': { # Uncomment if you want to run RF Classifier too
    #     'estimator': RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    #     'param_grid': {
    #         'n_estimators': [100, 200],
    #         'max_depth': [10, 20, None],
    #         'min_samples_leaf': [2, 4, 6],
    #         'max_features': ['sqrt', 'log2']
    #     }
    # },
    'LightGBM': {
        'estimator': lgb.LGBMClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [20, 31, 40],
            'max_depth': [-1, 10, 20]
        }
    },
    'XGBoost': {
        'estimator': xgb.XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='logloss'), # Added params for XGBoost
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.7, 0.8],
            'colsample_bytree': [0.7, 0.8]
        }
    }
}

print(f"Target threshold for 'Good Match': {GOOD_MATCH_THRESHOLD:.4f}")
print(f"Prepared configurations for {list(model_configs.keys())}")

Target threshold for 'Good Match': 0.8464
Prepared configurations for ['LightGBM', 'XGBoost']


### Step 4: Stateful Feature Engineering Function
This function applies transformations using parameters learned only from the training set.

In [24]:
def create_stateful_features(df, avg_adherence, min_dist, max_dist): 
    """Applies historical adherence and proximity score calculations."""
    df_eng = df.copy()
    
    # Historical Adherence
    df_sorted = df_eng.sort_values(by=['patient_id', 'encounter_date'])
    df_sorted['shifted_adherence'] = df_sorted.groupby('patient_id')['treatment_adherence'].shift(1)
    df_sorted['treatment_sum'] = df_sorted.groupby('patient_id')['shifted_adherence'].cumsum().fillna(0)
    df_sorted['treatment_count'] = df_sorted.groupby('patient_id').cumcount()
    df_sorted['historical_avg_adherence'] = np.where(
        df_sorted['treatment_count'] > 0,
        df_sorted['treatment_sum'] / df_sorted['treatment_count'],
        avg_adherence
    )
    # Use reindex to handle potential sorting changes
    df_eng['historical_avg_adherence'] = df_sorted['historical_avg_adherence'].reindex(df_eng.index)
    
    # Proximity Score
    df_eng['proximity_score'] = 1 - ((df_eng['distance_km'] - min_dist) / (max_dist - min_dist))
    df_eng['proximity_score'] = df_eng['proximity_score'].clip(0, 1)
    
    # Ensure all expected feature columns exist, fill missing numerical features if any
    for col in features:
        if col not in df_eng.columns:
             df_eng[col] = 0 # Or a more appropriate fill value
        elif pd.api.types.is_numeric_dtype(df_eng[col]):
            df_eng[col] = df_eng[col].fillna(df_eng[col].median()) # Example: fill numeric NaNs with median
            
    return df_eng[features] # Return only the final feature columns

### Step 5: Main Experiment Loop

In [25]:
# --- Dictionaries to Store All Results ---
all_results = {}
feature_engineering_params = {}

# --- Main Loop ---
unique_preferences = master_df['cultural_preferences'].unique()
for model_name, config in model_configs.items():
    print(f"\n{'='*20} RUNNING EXPERIMENT FOR MODEL: {model_name.upper()} {'='*20}")
    
    model_results = {
        'trained_models': {},
        'test_metrics': {},
        'best_params': {},
        'feature_weights': {}
    }
    
    for preference in unique_preferences:
        print(f"\n--- Training segment: '{preference}' ---")
        segment_df = master_df[master_df['cultural_preferences'] == preference].copy()
        
        if len(segment_df) < 100: continue
            
        # Create binary target for the segment
        segment_df[target] = (segment_df['success_score'] >= GOOD_MATCH_THRESHOLD).astype(int)
        
        # Split BEFORE stateful feature engineering
        X = segment_df.drop(columns=[target, 'success_score', 'satisfaction_norm', 'adherence_norm']) # Drop target and intermediate scores
        y = segment_df[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Stratify for classification
        
        # Learn stateful parameters from training data ONLY
        avg_adherence_train = X_train['treatment_adherence'].mean()
        min_dist_train = X_train['distance_km'].min()
        max_dist_train = X_train['distance_km'].max()
        feature_engineering_params[preference] = {'avg_adherence': avg_adherence_train, 'min_dist': min_dist_train, 'max_dist': max_dist_train}
        
        # Apply stateful feature engineering
        X_train_eng = create_stateful_features(X_train, avg_adherence_train, min_dist_train, max_dist_train)
        X_test_eng = create_stateful_features(X_test, avg_adherence_train, min_dist_train, max_dist_train)
        
        # --- Tuning --- 
        random_search = RandomizedSearchCV(
            estimator=config['estimator'],
            param_distributions=config['param_grid'],
            n_iter=20, cv=3, verbose=0, random_state=42, 
            scoring='roc_auc' # Use AUC for classification tuning
        )
        print(f'Tuning on {len(X_train_eng)} samples...')
        random_search.fit(X_train_eng, y_train)
        best_model = random_search.best_estimator_
        
        # --- Evaluation --- 
        print("Evaluating on test set...")
        # Predict probabilities for the positive class (Good Match)
        final_probs = best_model.predict_proba(X_test_eng)[:, 1] 
        # Use a threshold (e.g., 0.5) to make binary predictions for F1/Precision/Recall
        final_preds = (final_probs >= 0.5).astype(int)
        
        auc = roc_auc_score(y_test, final_probs)
        f1 = f1_score(y_test, final_preds)
        precision = precision_score(y_test, final_preds)
        recall = recall_score(y_test, final_preds)
        print(f"Test Set Metrics -> AUC: {auc:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
        print("\nClassification Report:\n", classification_report(y_test, final_preds))
        
        # --- Store results --- 
        model_results['trained_models'][preference] = best_model
        model_results['test_metrics'][preference] = {'auc': auc, 'f1': f1, 'precision': precision, 'recall': recall}
        model_results['best_params'][preference] = random_search.best_params_
        model_results['feature_weights'][preference] = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
        
    all_results[model_name] = model_results # Store results for the current model

print("\nAll experiments complete.")


==================== RUNNING EXPERIMENT FOR MODEL: LIGHTGBM ====================

--- Training segment: 'No Specific Preference' ---
Tuning on 69151 samples...
[LightGBM] [Info] Number of positive: 23234, number of negative: 22866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001870 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 918
[LightGBM] [Info] Number of data points in the train set: 46100, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Number of positive: 23235, number of negative: 22866
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000395 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_

c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:13:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:13:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:13:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:13:56] WARNING: C:\actions-r

Evaluating on test set...
Test Set Metrics -> AUC: 0.5266, F1: 0.5581, Precision: 0.5208, Recall: 0.6013

Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.44      0.48      8575
           1       0.52      0.60      0.56      8713

    accuracy                           0.52     17288
   macro avg       0.52      0.52      0.52     17288
weighted avg       0.52      0.52      0.52     17288


--- Training segment: 'Culturally Similar Provider' ---
Tuning on 55405 samples...


c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:11] WARNING: C:\actions-r

Evaluating on test set...
Test Set Metrics -> AUC: 0.5213, F1: 0.6470, Precision: 0.5143, Recall: 0.8720

Classification Report:
               precision    recall  f1-score   support

           0       0.51      0.14      0.22      6779
           1       0.51      0.87      0.65      7073

    accuracy                           0.51     13852
   macro avg       0.51      0.51      0.43     13852
weighted avg       0.51      0.51      0.44     13852


--- Training segment: 'Culturally Similar Provider; Same Language Provider' ---
Tuning on 15664 samples...


c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:14:24] WARNING: C:\actions-r

Evaluating on test set...
Test Set Metrics -> AUC: 0.9876, F1: 0.0000, Precision: 0.0000, Recall: 0.0000

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.99      3816
           1       0.00      0.00      0.00       100

    accuracy                           0.97      3916
   macro avg       0.49      0.50      0.49      3916
weighted avg       0.95      0.97      0.96      3916


--- Training segment: 'Same Language Provider' ---
Tuning on 19779 samples...


c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\jerry\anaconda3\envs\Env2709_Capstone_py_311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

Evaluating on test set...
Test Set Metrics -> AUC: 0.9867, F1: 0.5677, Precision: 0.5409, Recall: 0.5972

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.99      4801
           1       0.54      0.60      0.57       144

    accuracy                           0.97      4945
   macro avg       0.76      0.79      0.78      4945
weighted avg       0.97      0.97      0.97      4945


All experiments complete.


### Step 6: Consolidate Results and Save Artifacts

In [26]:
# Create a summary DataFrame for easy comparison
summary_list = []
for model_name, results in all_results.items():
    for preference, metrics in results['test_metrics'].items():
        row = {
            'model_name': model_name,
            'preference_group': preference,
            'auc': metrics['auc'],
            'f1': metrics['f1'],
            'precision': metrics['precision'],
            'recall': metrics['recall']
        }
        summary_list.append(row)
summary_df = pd.DataFrame(summary_list)

print("--- Final Performance Comparison (AUC) ---")
display(summary_df.sort_values(by=['preference_group', 'auc'], ascending=[True, False]))

# --- Save artifacts (models, parameters, feature engineering params) ---
artifacts_to_save = {
    'all_results': all_results, # Contains models, metrics, params, weights
    'feature_engineering_params': feature_engineering_params # Contains min/max dist, avg adherence per segment
}

file_path = 'classification_model_artifacts.joblib'
joblib.dump(artifacts_to_save, file_path)
print(f"\nAll models and results saved to '{file_path}'")

--- Final Performance Comparison (AUC) ---


,model_name,preference_group,auc,f1,precision,recall
5,XGBoost,Culturally Similar Provider,0.521348,0.647050,0.514343,0.872049
1,LightGBM,Culturally Similar Provider,0.517468,0.435945,0.532544,0.369009
2,LightGBM,Culturally Similar Provider; Same Language Pro...,0.988762,0.699647,0.540984,0.990000
6,XGBoost,Culturally Similar Provider; Same Language Pro...,0.987649,0.000000,0.000000,0.000000
0,LightGBM,No Specific Preference,0.527195,0.515934,0.522928,0.509124
4,XGBoost,No Specific Preference,0.526602,0.558142,0.520775,0.601285
3,LightGBM,Same Language Provider,0.987059,0.704156,0.543396,1.000000
7,XGBoost,Same Language Provider,0.986688,0.567657,0.540881,0.597222



All models and results saved to 'classification_model_artifacts.joblib'


### Step 7: Define the Recommendation Function

In [ ]:
def get_classification_recommendations(patient_id, required_specialty, 
                                       # DataFrames
                                       all_providers_df, all_patients_df, all_hospitals_df,
                                       # Loaded Artifacts
                                       all_results_loaded, feature_eng_params_loaded,
                                       # Top N recommendations
                                       top_n=5):
    """
    Generates ranked doctor recommendations using the trained classification models.
    Ranks based on the predicted probability of a 'Good Match'.
    """
    print(f"\n--- Generating recommendations for Patient ID: {patient_id} ---")
    patient_info = all_patients_df[all_patients_df['patient_id'] == patient_id]
    if patient_info.empty: return "Error: Patient ID not found."
        
    preference = patient_info['cultural_preferences'].iloc[0]
    model_name_to_use = None # We'll select the best model based on AUC later
    segment_params = feature_eng_params_loaded.get(preference)
    
    if not segment_params:
        # Fallback if the segment was too small to train
        print(f"Warning: No specific model found for preference '{preference}'. Using fallback logic or a default model if available.")
        # Handle fallback (e.g., use 'No Specific Preference' model if it exists)
        preference = 'No Specific Preference' # Example fallback
        segment_params = feature_eng_params_loaded.get(preference)
        if not segment_params:
            return "Error: Cannot generate recommendations, no suitable model artifacts found."

    # --- Select the best performing model for this segment (based on AUC stored in results) ---
    best_auc = -1
    for model_key, results in all_results_loaded.items():
        if preference in results['test_metrics']:
            current_auc = results['test_metrics'][preference]['auc']
            if current_auc > best_auc:
                best_auc = current_auc
                model_name_to_use = model_key
                
    if not model_name_to_use:
         return f"Error: No successfully trained model found for preference '{preference}'."
         
    model_to_use = all_results_loaded[model_name_to_use]['trained_models'].get(preference)
    print(f"Using model '{model_name_to_use}' for preference '{preference}' (AUC: {best_auc:.4f}).")

    # --- Candidate Generation (same as before) ---
    candidate_providers = all_providers_df[(all_providers_df['specialty'] == required_specialty) & (all_providers_df['accepts_new_patients'] == True)].copy()
    if candidate_providers.empty: return f"Error: No providers found for specialty '{required_specialty}' accepting new patients."

    # --- Feature Engineering for Inference ---
    inference_df = candidate_providers.assign(key=1).merge(patient_info.assign(key=1), on='key').drop('key', axis=1)
    inference_df = pd.merge(inference_df, all_hospitals_df, left_on='hospital_affiliation', right_on='hospital_id', how='left')
    inference_df.rename(columns={'cultural_competency_rating_y': 'cultural_competency_rating_prov'}, inplace=True)
    inference_df['race_match'] = (inference_df['race'] == inference_df['provider_race']).astype(int)
    inference_df['ethnicity_match'] = (inference_df['ethnicity'] == inference_df['provider_ethnicity']).astype(int)
    inference_df['language_match'] = (inference_df['language_match_x'] == True).astype(int)
    inference_df['distance_km'] = pgeocode.GeoDistance('US').query_postal_code(inference_df['zip_code'].astype(str).tolist(), inference_df['zip_code_hosp'].astype(str).tolist())
    mean_dist_by_specialty = inference_df.groupby('specialty')['distance_km'].transform('mean')
    inference_df['distance_km'].fillna(mean_dist_by_specialty, inplace=True)
    inference_df['distance_km'].fillna(segment_params['max_dist'] / 2, inplace=True)
    
    # Apply stateful features using LOADED parameters
    X_inference_eng = create_stateful_features(inference_df, 
                                              segment_params['avg_adherence'], 
                                              segment_params['min_dist'], 
                                              segment_params['max_dist'])

    # --- Predict Probabilities --- 
    predicted_probabilities = model_to_use.predict_proba(X_inference_eng)[:, 1] # Probability of class 1 ('Good Match')
    inference_df['predicted_prob_good_match'] = predicted_probabilities

    # --- Rank and Return --- 
    recommendations = inference_df.sort_values(by='predicted_prob_good_match', ascending=False)
    
    print("--- Recommendations Generated (Ranked by Probability of Good Match) ---")
    return recommendations[['provider_id', 'first_name_x', 'last_name_x', 'specialty', 'predicted_prob_good_match']].head(top_n)

### Step 8: Example Inference Call (Loading Artifacts)

In [ ]:
# --- Load artifacts in a separate session/script ---
file_path = 'classification_model_artifacts.joblib'
loaded_artifacts = joblib.load(file_path)

all_results_loaded = loaded_artifacts['all_results']
feature_eng_params_loaded = loaded_artifacts['feature_engineering_params']

print("--- Artifacts loaded successfully ---")

# --- Example Usage ---
example_patient_id = 'PAT_046599' # Replace with valid ID
example_specialty = 'Cardiology' # Replace with valid specialty

final_recommendations_clf = get_classification_recommendations(
    patient_id=example_patient_id, 
    required_specialty=example_specialty, 
    all_providers_df=provider_df, 
    all_patients_df=patient_df, 
    all_hospitals_df=hospital_df,
    all_results_loaded=all_results_loaded,
    feature_eng_params_loaded=feature_eng_params_loaded,
    top_n=5
)

display(final_recommendations_clf)